In [377]:
import os
import sys 
os.chdir("/workspaces/dev/app")
sys.path.append("/workspaces/dev/app")

In [378]:
from services.whisper import WordService, WordParams, WordReturn
from services.whisper.Params import Hyperparameters
import librosa
import numpy as np
from silero_vad import load_silero_vad, get_speech_timestamps

In [379]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [380]:
audio, sr = librosa.load("/workspaces/dev/.data/news_with_english.mp3", sr=SAMPLE_RATE)

In [381]:
# audio = audio[85 * SAMPLE_RATE:]

In [382]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=32000, scale=400))
  rand_len = np.clip(rand_len, 28000, 36000)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [383]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [384]:
# print(full_text)

In [385]:
HYPERPARAMETERS = {
  "weighted_prob_boundary": 0,
  "filter_by_duration_z": {
    "default": 2.0,
    "ko": 1.5,
    "en": 2.0,
  },
  "filter_by_probability": {
    "z": {
      "default": 2.0,
      "ko": 2.0,
      "en": 2.0,
    },
    "min_prob": {
      "default": 1.0,
      "ko": 0.4,
      "en": 0.4,
    },
  },
  "token_iou_padding": 0.2,
  "combine": {
    "search_range_time": {
      "default": 1.5,
      "ko": 1.5,
      "en": 1.5,
    },
    "threshold": {
      "default": 0.5,
      "ko": 0.25,
      "en": 0.5,
    },
    "tolerance": {
      "default": 0.3,
      "ko": 0.3,
      "en": 0.3,
    },
  },
  "refine_tolerance": {
    "default": 0.5,
    "ko": 0.5,
    "en": 0.5,
  }
}

In [386]:
MAX_PREV_TIME = 5

In [387]:
class TestWordService(WordService):
  def _get_weighted_probability(self, probabilities, start, end, duration, boundary):
    center = (start + end) / 2
    if center > boundary:
      if center < duration - boundary:
        return probabilities
      return probabilities - probabilities * ((boundary - duration + center)/boundary) ** 2
    return probabilities - probabilities * (boundary - center/boundary) ** 2

In [388]:
hyper = Hyperparameters(None, HYPERPARAMETERS)

In [389]:
whisper_service = TestWordService.get_instance(MAX_PREV_TIME, hyper)

In [390]:
model = load_silero_vad(onnx=True)

In [391]:
raise Exception("stop")

Exception: stop

In [392]:
def vad_audio(audio):
  timestamps = get_speech_timestamps(
    audio,
    model,
    sampling_rate=SAMPLE_RATE,
    threshold=0.4,
    min_silence_duration_ms = 400,
    speech_pad_ms=300
  )

  merged_audio = []

  for segment in timestamps:
    start = segment['start']
    end = segment['end']
    merged_audio.append(audio[start:end])

  return np.concatenate(merged_audio)


In [393]:
from IPython.display import Audio

In [427]:
segment_id = 0
completed = {}
word_params = WordParams()

In [442]:
segment = segments[segment_id]
segment_id += 1

word_params.audio = vad_audio(segment)

(result, audio) = whisper_service.transcribe(word_params)
completed.update(result.completed_dict)

print(f"{segment_id}" + "--" * 20)
print([(v.lang, v.text) for k, v in completed.items()])
print([(v.lang, v.text) for v in result.prev_words if v.is_word])
print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

word_params.order = result.order
word_params.time_offset = result.time_offset
word_params.prev_audio = result.prev_audio
word_params.prev_words = result.prev_words
word_params.prev_recog = result.prev_recog
word_params.prev_prob_mean = result.prev_prob_mean
word_params.prev_prob_std = result.prev_prob_std
word_params.prev_prob_count = result.prev_prob_count
word_params.prev_dura_mean = result.prev_dura_mean
word_params.prev_dura_std = result.prev_dura_std
word_params.prev_dura_count = result.prev_dura_count

Audio(audio, rate=SAMPLE_RATE)

15----------------------------------------
[(['ko'], 'BBC 생방송 인터뷰도 중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.'), (['ko'], '유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 등장했습니다.'), (['ko'], '배영진 기자입니다.'), (['ko'], 'BBC 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리, 부산대 교수.')]
[('ko', ' 유튜브'), ('ko', ' 영상')]
[('ko', ' 조회수가'), ('ko', ' 감사합니다.'), ('ko', ' 건이'), ('ko', ' 넘는'), ('ko', ' 켈리'), ('ko', ' 교수'), ('ko', ' 가족은'), ('ko', ' 세계적인'), ('ko', ' 유명인사가'), ('ko', ' 됐습니다.')]


In [ ]:
Audio(result.prev_audio, rate=SAMPLE_RATE)

In [395]:
for segment in segments:
  word_params.audio = segment

  (result, audio) = whisper_service.transcribe(word_params)
  completed.update(result.completed_dict)

  print(f"{segment_id}" + "--" * 20)
  print([(v.lang, v.text) for k, v in completed.items()])
  print([(v.lang, v.text) for v in result.prev_words if v.is_word])
  print([(v.lang, v.text) for v in result.prev_recog if v.is_word])

  word_params.order = result.order
  word_params.time_offset = result.time_offset
  word_params.prev_audio = result.prev_audio
  word_params.prev_words = result.prev_words
  word_params.prev_recog = result.prev_recog
  word_params.prev_prob_mean = result.prev_prob_mean
  word_params.prev_prob_std = result.prev_prob_std
  word_params.prev_prob_count = result.prev_prob_count
  word_params.prev_dura_mean = result.prev_dura_mean
  word_params.prev_dura_std = result.prev_dura_std
  word_params.prev_dura_count = result.prev_dura_count

0----------------------------------------
[]
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도')]
0----------------------------------------
[]
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에'), ('ko', ' 자녀'), ('ko', ' 난입사건.')]
0----------------------------------------
[]
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에'), ('ko', ' 자녀'), ('ko', ' 난입'), ('ko', ' 사건으로'), ('ko', ' 스타가'), ('ko', ' 된'), ('ko', ' 미국인'), ('ko', ' 교수가')]
0----------------------------------------
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에')]
[('ko', ' 자녀'), ('ko', ' 난입'), ('ko', ' 사건으로'), ('ko', ' 스타가'), ('ko', ' 된'), ('ko', ' 미국인'), ('ko', ' 교수'), ('ko', ' 가족이'), ('ko', ' 오늘'), ('ko', ' 카메라'), ('ko', ' 앞에'), ('ko', ' 섰습니다.')]
0----------------------------------------
[]
[('ko', ' BBC'), ('ko', ' 생방송'), ('ko', ' 인터뷰도'), ('ko', ' 도중에'), ('ko', ' 자녀'), ('ko', ' 난입'), ('ko', ' 사건으로'), ('ko', ' 스타가')]
[('ko', ' 된'), ('ko', ' 미국인'), ('ko', ' 교수'), ('ko', ' 가족이

In [396]:
for key, item in completed.items():
  print(key, item)
for v in result.prev_words:
  if v.is_word: print(v.text)
# for v in prev_recog:
#   print(v.text)

0 ['ko'] BBC 생방송 인터뷰도 도중에 자녀 난입 사건으로 스타가 된 미국인 교수 가족이 오늘 카메라 앞에 섰습니다.
1 ['ko'] 유튜브 스타가 된 4살짜리 딸은 이번엔 사탕을 입에 물고 배영진 기자입니다.
2 ['ko'] BBC 인터뷰 도중 딸과 아들의 등장으로 일약 스타가 된 로버트 켈리, 부산대 교수.
3 ['ko'] 유튜브 영상 감사합니다.
4 ['ko'] 건이 넘는 켈리 교수 가족은 세계적인 유명인사가 됐습니다.
5 ['ko'] 이렇게 언론의 관심이 커지자 켈리 교수 가족이 기자회견에 나섰습니다.
6 ['ko'] 춤을 췄던 첫째 딸 메리아는 사탕을 물었고 둘째 아들 존은 엄마 품에 안긴 모습이었습니다. 
7 ['en'] thought it was a disaster. 
8 ['en'] I immediately called texted or or emailed the BBC. 
9 ['en'] I communicated with the BBC immediately afterwards, and I apologized to them. 
10 ['en'] I said that if they never us back or never asked me to be on television again.
11 ['en'] I would understand.
12 ['ko'] 귀여운 춤으로 화제가 된 딸에 대한 질문도 잇따랐습니다.
13 ['en'] answered, that's She's four.
14 ['en'] She has no idea.
15 ['ko'] 당시 BBC와 인터뷰한 내용은 한국의 대통령 탄핵 사건이었습니다. 
16 ['en', 'ko'] months months millions of people on the streets, no one's car got burned The protesters even picked up their trash I find that that's just a model of a 일부 외국 네티즌들이 켈리 교수의